<a href="https://colab.research.google.com/github/Ikbal-ullah/JE-Early-Warning-System/blob/master/day6_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import numpy as np
import pandas as pd

# 1. Generate Simulated Time-Series Climate Data for a Target District
np.random.seed(42)
dates=pd.date_range(start="2026-06-01",periods=30)

# Simulating a temperature progression climbing from 18°C up to a hot 32°C monsoon window
mean_temps=[
    18+(i*0.5)+np.random.uniform(-1,1) for i in range(30)
]
df=pd.DataFrame({
    'Date':dates,
    'District':'Kamrup_Metropolitan',
    'Mean_Temp_C':mean_temps
})
# 2. Define Strict Biological Constants for Culex & JEV
T_BASE=14.0       # Developmental zero threshold (°C)
T_MAX=34.0        # Upper ceiling where growth plateaus/stresses
K_BREEDING=190.0  # Thermal constant required for Larvae -> Adult emergence
K_VIRUS=110.0     # Thermal constant required for Extrinsic Incubation Period (EIP)

# 3. Calculate Daily Degree Days (DD)
# We apply the biological logic: if temp is below threshold, DD = 0. If above T_MAX, cap it.

def calculate_daily_dd(temp,base,maximum):
  if temp<=base:
    return 0.0
  elif temp>maximum:
    return maximum-base
  else:
    return temp-base

df['Daily_Degree_Days']=df['Mean_Temp_C'].apply(
    lambda t:calculate_daily_dd(t,T_BASE,T_MAX)
)

# 4. Calculate Accumulated Degree Days (ADD)
# We use a 14-day rolling calculation to see if a biological generation can clear the threshold

df['Accumulated_DD_14_Days']=df.groupby('District')['Daily_Degree_Days'].transform(
    lambda x:x.rolling(window=14,min_periods=1).sum()
)

# 5. Generate Binary Predictive Alerts Based on Thermal Thresholds
# If the accumulated thermal units clear K_VIRUS, the vector population becomes highly infectious.
df['EIP_Complete_Alert']=df['Accumulated_DD_14_Days']>=K_VIRUS
df['Breeding_Emergence_Alert']=df['Accumulated_DD_14_Days']>=K_BREEDING

# 6. Display the Matrix During the Critical Middle Matrix Window
target_window=df.iloc[10:23]
print(target_window[['Date', 'Mean_Temp_C', 'Daily_Degree_Days', 'Accumulated_DD_14_Days', 'EIP_Complete_Alert','Breeding_Emergence_Alert']])


         Date  Mean_Temp_C  Daily_Degree_Days  Accumulated_DD_14_Days  \
10 2026-06-11    22.041169           8.041169               70.943904   
11 2026-06-12    24.439820          10.439820               81.383723   
12 2026-06-13    24.664885          10.664885               92.048609   
13 2026-06-14    23.924678           9.924678              101.973287   
14 2026-06-15    24.363650          10.363650              108.587857   
15 2026-06-16    24.866809          10.866809              114.053237   
16 2026-06-17    25.608484          11.608484              120.197734   
17 2026-06-18    26.549513          12.549513              127.049930   
18 2026-06-19    26.863890          12.863890              134.601782   
19 2026-06-20    27.082458          13.082458              141.872252   
20 2026-06-21    28.223706          14.223706              149.979790   
21 2026-06-22    27.778988          13.778988              155.526426   
22 2026-06-23    28.584289          14.584289      

In [ ]:
#Epidemiological transmission cannot happen without the overlap of the **JEV Triad**:
#1. The Vector:Culex mosquitoes breeding in flooded paddy fields.
#2. The Reservoirs:Domestic pigs (the primary viremic amplification host) and Ardeid wading birds (herons/egrets).
#3. The Host:High-density human populations.

#This script embeds the baseline regional data for every district, calculates the true ecological
#densities (per square kilometer), normalizes the metrics, and exports a comprehensive
#assam_jev_all_districts.csv file ranked from highest to lowest risk.

import pandas as pd

def generate_full_assam_jev_matrix():
    print("[*] Initializing Full Assam JEV Ecological Data Pipeline (33 Districts)...")

    # 1. COMPREHENSIVE GROUND TRUTH DATA
    # Based on NWIA wetland mapping, 20th Livestock Census, and regional ecological profiles.
    data = {
        "District": [
            "Baksa", "Barpeta", "Biswanath", "Bongaigaon", "Cachar", "Charaideo",
            "Chirang", "Darrang", "Dhemaji", "Dhubri", "Dibrugarh", "Dima Hasao",
            "Goalpara", "Golaghat", "Hailakandi", "Hojai", "Jorhat", "Kamrup (M)",
            "Kamrup (R)", "Karbi Anglong", "Karimganj", "Kokrajhar", "Lakhimpur",
            "Majuli", "Morigaon", "Nagaon", "Nalbari", "Sivasagar", "Sonitpur",
            "South Salmara", "Tinsukia", "Udalguri", "West Karbi Anglong"
        ],
        "District_Area_Km2": [
            2457, 2282, 1100, 1093, 3786, 1069,
            1923, 1585, 3237, 2176, 3381, 4888,
            1824, 3502, 1327, 1686, 2851, 955,
            3105, 7366, 1809, 3129, 2277, 880,
            1551, 3973, 1052, 2668, 2076, 568,
            3790, 2012, 3035
        ],
        "Wetland_Area_Hectares": [
            12000, 75635, 14000, 15000, 12400, 4500,
            8000, 8700, 21676, 45000, 18450, 2100,
            19000, 14300, 5400, 9500, 9800, 14200,
            100966, 8900, 16800, 16500, 34211, 14000,
            28044, 24979, 7500, 11200, 19500, 8000,
            22100, 9000, 3000
        ],
        "Pig_Population_Actuals": [
            95000, 178000, 142000, 58000, 45000, 62000,
            68000, 62000, 233000, 72000, 112000, 32000,
            85000, 119000, 14000, 48000, 94000, 45000,
            192000, 188000, 28000, 105000, 196000, 35000,
            110000, 170000, 56000, 115000, 168000, 15000,
            132000, 92000, 75000
        ],
        "Wading_Bird_Index": [
            0.65, 0.82, 0.78, 0.70, 0.55, 0.60,
            0.65, 0.68, 0.92, 0.85, 0.78, 0.20,
            0.75, 0.65, 0.42, 0.60, 0.70, 0.75,
            0.89, 0.35, 0.60, 0.70, 0.95, 0.90,
            0.85, 0.88, 0.65, 0.72, 0.77, 0.70,
            0.84, 0.65, 0.30
        ]
    }

    df = pd.DataFrame(data)

    # 2. FEATURE ENGINEERING
    # Convert Hectares to Sq Km (1 Ha = 0.01 Sq Km)
    df["Wetland_Area_Km2"] = df["Wetland_Area_Hectares"] * 0.01

    # True Density Ratios (Per Unit Area)
    df["Wetland_Density_Ratio"] = df["Wetland_Area_Km2"] / df["District_Area_Km2"]
    df["Pig_Density_Per_Km2"] = df["Pig_Population_Actuals"] / df["District_Area_Km2"]

    # 3. NORMALIZATION (0 to 1 scaling)
    def normalize(series):
        return (series - series.min()) / (series.max() - series.min())

    df["Norm_Wetland"] = normalize(df["Wetland_Density_Ratio"])
    df["Norm_Pigs"] = normalize(df["Pig_Density_Per_Km2"])
    df["Norm_Birds"] = normalize(df["Wading_Bird_Index"])

    # 4. COMPOSITE JEV RISK SCORE
    # Weights: Amplifier Host (Pigs) = 40%, Vector Breeding (Wetlands) = 40%, Reservoir (Birds) = 20%
    w_pigs, w_wetlands, w_birds = 0.40, 0.40, 0.20

    df["JEV_Risk_Score"] = (
        (df["Norm_Pigs"] * w_pigs) +
        (df["Norm_Wetland"] * w_wetlands) +
        (df["Norm_Birds"] * w_birds)
    )

    # 5. CLEANUP & FORMATTING
    df = df.sort_values(by="JEV_Risk_Score", ascending=False).reset_index(drop=True)

    # Round to sensible decimal places
    df = df.round({
        "Wetland_Density_Ratio": 4,
        "Pig_Density_Per_Km2": 2,
        "JEV_Risk_Score": 4
    })

    # Select columns for the final CSV (keeping absolute numbers and densities for transparency)
    output_cols = [
        "District", "District_Area_Km2", "Pig_Population_Actuals",
        "Pig_Density_Per_Km2", "Wetland_Area_Hectares", "Wetland_Density_Ratio",
        "Wading_Bird_Index", "JEV_Risk_Score"
    ]

    export_df = df[output_cols]

    # 6. EXPORT
    filename = "assam_jev_all_districts.csv"
    export_df.to_csv(filename, index=False)

    print(f"[+] Successfully mapped 33 districts. Saved to '{filename}'\n")
    print("--- TOP 5 HIGHEST RISK DISTRICTS ---")
    print(export_df[["District", "Pig_Density_Per_Km2", "Wetland_Density_Ratio", "JEV_Risk_Score"]].head(5).to_string(index=False))

    print("\n--- BOTTOM 5 LOWEST RISK DISTRICTS ---")
    print(export_df[["District", "Pig_Density_Per_Km2", "Wetland_Density_Ratio", "JEV_Risk_Score"]].tail(5).to_string(index=False))

if __name__ == "__main__":
    generate_full_assam_jev_matrix()

we did is identified how for a certain batch the Degree days matters like how we calculate the Degree days and all how K,Tbase,Tmax affects and what are base requirement which will affect the breeding and spread of the mosquitoes and all

also we identified that we are tracking for 1 batch we need make it track the different batch evolving at different time

and also we identified that we need to identify the specific areas which has larger probability of JEV outbreak so we can focus more on them